# Multimodal Deep Learning for Autoimmune Disease Diagnosis
## RA Clinical / Metadata Branch — Google Colab Notebook

**Scope:** Rheumatoid Arthritis (RA) only. This notebook trains and evaluates the **clinical/metadata branch** only.

The X-ray branch (ImageNet-pretrained ResNet18) is assumed to have been trained separately. **No image features are used here and no multimodal fusion is performed here.**

### Purpose
Train a clinical-data model using patient-level metadata associated with RA/non-RA cases, evaluate it without patient leakage, and save processed clinical representations for future multimodal fusion.

> **Important repository check:** the configured GitHub repository is inspected at runtime before any data assumptions are made. The notebook never invents metadata columns, labels, paths, or clinical variables.

## 1 — Project Introduction

In [ ]:
PROJECT_TITLE = "Multimodal Deep Learning for Autoimmune Disease Diagnosis"
BRANCH = "RA Clinical / Metadata"
print(PROJECT_TITLE)
print("Current branch:", BRANCH)
print("RA only: YES")
print("X-ray training in this notebook: NO")
print("Multimodal fusion in this notebook: NO")

## 2 — Environment Setup

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib openpyxl

In [ ]:
import os, sys, json, random, platform, warnings, subprocess, shutil, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("Seed:", SEED)

## 3 — Clone / Access GitHub Repository

In [ ]:
REPO_URL = "https://github.com/Varshini-3012/Autoimmune.git"
WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "Autoimmune"

if REPO_DIR.exists():
    print("Repository directory already exists:", REPO_DIR)
else:
    print("Cloning:", REPO_URL)
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Git clone failed. Check repository availability/network access.")

all_files = [p for p in REPO_DIR.rglob("*") if p.is_file() and ".git" not in p.parts]
print(f"Repository files found: {len(all_files)}")
for p in all_files[:200]:
    print(p.relative_to(REPO_DIR))

if len(all_files) == 0:
    raise RuntimeError(
        "The configured GitHub repository contains no project/data files at runtime. "
        "This notebook will NOT fabricate a metadata path or silently substitute another dataset. "
        "Add the RA dataset/metadata to the repository (or change REPO_URL to the correct repository) and rerun."
    )

## 4 — Inspect Dataset and Metadata

In [ ]:
TABULAR_EXTS = {".csv", ".xlsx", ".xls", ".parquet", ".json"}
tabular_files = [p for p in all_files if p.suffix.lower() in TABULAR_EXTS]

print("Candidate metadata/tabular files:")
for i, p in enumerate(tabular_files):
    print(f"[{i}] {p.relative_to(REPO_DIR)}")

if not tabular_files:
    raise FileNotFoundError(
        "No CSV/XLSX/XLS/Parquet/JSON metadata files were found. "
        "Available repository files were printed above."
    )

def load_table(path):
    ext = path.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext == ".json":
        return pd.read_json(path)
    raise ValueError(ext)

profiles = []
for p in tabular_files:
    try:
        df0 = load_table(p)
        profiles.append({
            "path": p,
            "rows": len(df0),
            "cols": len(df0.columns),
            "columns": list(map(str, df0.columns))
        })
    except Exception as e:
        print("Could not inspect", p, "->", repr(e))

for x in profiles:
    print("\nFILE:", x["path"].relative_to(REPO_DIR))
    print("Shape:", (x["rows"], x["cols"]))
    print("Columns:", x["columns"])

In [ ]:
# Candidate selection is evidence-based: prefer tables whose columns indicate RA label + patient identity.
def score_metadata_profile(cols):
    low = [c.lower().strip() for c in cols]
    score = 0
    reasons = []
    if any(c in low for c in ["isra", "is_ra", "ra", "label", "target", "diagnosis"]):
        score += 5; reasons.append("possible target column")
    if any(("patient" in c) or c in {"studyid", "subjectid", "subject_id"} for c in low):
        score += 4; reasons.append("possible patient identifier")
    if any(c in low for c in ["age", "sex", "gender"]):
        score += 3; reasons.append("clinical demographic columns")
    if any("image" in c or "stem" in c or "file" in c for c in low):
        score += 1; reasons.append("image linkage column")
    return score, reasons

ranked = []
for x in profiles:
    score, reasons = score_metadata_profile(x["columns"])
    ranked.append((score, x["path"], reasons, x["columns"]))
ranked.sort(key=lambda z: z[0], reverse=True)

print("Ranked metadata candidates:")
for score, path, reasons, cols in ranked:
    print(f"{score:2d} | {path.relative_to(REPO_DIR)} | {', '.join(reasons)}")

if not ranked or ranked[0][0] == 0:
    raise RuntimeError("No metadata file could be identified safely from actual column names.")

METADATA_PATH = ranked[0][1]
df = load_table(METADATA_PATH)

print("\nSELECTED METADATA:", METADATA_PATH.relative_to(REPO_DIR))
print("Reason: highest evidence score based on actual target/patient/clinical/linkage column names.")
print("Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing counts:")
display(df.isna().sum().to_frame("missing"))
print("\nUnique counts:")
display(df.nunique(dropna=False).to_frame("unique"))
print("\nNumerical summary:")
display(df.describe(include=[np.number]).T)

In [ ]:
# Show sample categorical values without assuming their meaning.
cat_like = [c for c in df.columns if (df[c].dtype == "object" or str(df[c].dtype).startswith("category") or df[c].nunique(dropna=True) <= 20)]
for c in cat_like:
    vals = df[c].dropna().astype(str).unique()[:15]
    print(f"{c}: {vals}")

## 5 — Define Clinical Features

In [ ]:
# Identify columns conservatively from actual names. If critical columns cannot be identified, STOP.
cols = list(df.columns)
norm = {c: re.sub(r"[^a-z0-9]+", "", str(c).lower()) for c in cols}

def first_match(exact=(), contains=()):
    for c, n in norm.items():
        if n in exact:
            return c
    for c, n in norm.items():
        if any(k in n for k in contains):
            return c
    return None

PATIENT_COL = first_match(
    exact={"normalizedpatientid","patientid","subjectid","studyid"},
    contains={"patientid","subjectid"}
)
TARGET_COL = first_match(
    exact={"isra","ra","ralabel","target","label"},
    contains={"isra","ralabel"}
)
IMAGE_COL = first_match(
    exact={"mappedimagestem","imagestem","imageid","filename","filepath"},
    contains={"imagestem","imagefile","filename","filepath"}
)

print("Detected patient ID:", PATIENT_COL)
print("Detected target:", TARGET_COL)
print("Detected image/file identifier:", IMAGE_COL)

if PATIENT_COL is None or TARGET_COL is None:
    raise RuntimeError(
        "Could not safely identify required patient ID and/or RA target. "
        f"Actual columns are: {cols}. Update only the detection rules after verifying column meaning."
    )

# Candidate clinical variables: strict allow-list by semantic name.
clinical_candidates = []
for c, n in norm.items():
    if n in {"age", "sex", "gender"}:
        clinical_candidates.append(c)

print("Clinical candidates found from actual metadata:", clinical_candidates)

if not clinical_candidates:
    raise RuntimeError(
        "No clearly clinical variables (e.g., age/sex/gender) were found. "
        "The notebook will not treat image acquisition fields, IDs, or unclear columns as clinical predictors."
    )

identifier_cols = [c for c in [PATIENT_COL, IMAGE_COL] if c is not None]
for c, n in norm.items():
    if any(k in n for k in ["path", "filename", "imagestem", "imageid", "caseid"]) and c not in identifier_cols:
        identifier_cols.append(c)

CLINICAL_FEATURES = [c for c in clinical_candidates if c not in identifier_cols and c != TARGET_COL]

print("Identifier columns excluded:", identifier_cols)
print("Target:", TARGET_COL)
print("Final clinical feature list:", CLINICAL_FEATURES)

## 6 — Patient-Level Data Handling

In [ ]:
# Inspect consistency within patients before collapsing duplicates.
def nunique_nonnull(s):
    return s.dropna().nunique()

consistency = df.groupby(PATIENT_COL)[CLINICAL_FEATURES + [TARGET_COL]].agg(nunique_nonnull)
problem_patients = consistency[(consistency > 1).any(axis=1)]
print("Patients with conflicting repeated clinical/target values:", len(problem_patients))
if len(problem_patients):
    display(problem_patients.head(20))
    raise RuntimeError(
        "Conflicting patient-level clinical/target values were found. "
        "Resolve these conflicts explicitly before patient-level aggregation."
    )

print("Repeated image rows can be collapsed safely for the verified clinical/target columns.")

## 7 — Clinical Dataset Construction

In [ ]:
# One record per patient: first non-null value is safe only after the consistency check above.
def first_nonnull(s):
    z = s.dropna()
    return z.iloc[0] if len(z) else np.nan

patient_df = (
    df.groupby(PATIENT_COL, as_index=False)
      .agg({**{c:first_nonnull for c in CLINICAL_FEATURES},
            TARGET_COL:first_nonnull})
)

print("Image/metadata rows:", len(df))
print("Unique patient records:", len(patient_df))
display(patient_df.head())

# Validate/normalize binary target using observed values only.
print("Observed target values:", patient_df[TARGET_COL].value_counts(dropna=False).to_dict())

def normalize_binary_target(s):
    nonnull = s.dropna()
    vals = list(pd.unique(nonnull))
    # Common explicit RA encodings only; otherwise stop.
    if set(vals).issubset({0,1,False,True}):
        return s.astype(int)
    lowmap = {str(v).strip().lower(): v for v in vals}
    if set(lowmap).issubset({"ra","non-ra","nonra","rheumatoid arthritis","control","healthy"}):
        return s.astype(str).str.strip().str.lower().map(
            {"ra":1, "rheumatoid arthritis":1, "non-ra":0, "nonra":0, "control":0, "healthy":0}
        )
    raise RuntimeError(f"Target encoding is not safely recognized. Observed values: {vals}")

patient_df["_target"] = normalize_binary_target(patient_df[TARGET_COL])
if patient_df["_target"].isna().any():
    raise RuntimeError("Missing/unknown target labels remain after normalization.")

print("RA patients:", int((patient_df["_target"]==1).sum()))
print("Non-RA patients:", int((patient_df["_target"]==0).sum()))

ax = patient_df["_target"].map({0:"Non-RA",1:"RA"}).value_counts().reindex(["Non-RA","RA"]).plot(kind="bar")
ax.set_title("Patient-level class distribution")
ax.set_ylabel("Patients")
plt.show()

## 8 — Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    "missing_count": patient_df[CLINICAL_FEATURES].isna().sum(),
    "missing_percent": patient_df[CLINICAL_FEATURES].isna().mean()*100
})
display(missing)
print("Imputation will be fitted ONLY on training data inside the preprocessing pipeline.")

## 9 — Patient-Level Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# 70/15/15 patient split, stratified when mathematically possible.
y_all = patient_df["_target"]

if y_all.value_counts().min() < 3:
    raise RuntimeError(
        "At least one class has fewer than 3 patients, so a reliable stratified train/val/test split "
        "cannot be created without empty/unstable class representation."
    )

train_df, temp_df = train_test_split(
    patient_df, test_size=0.30, random_state=SEED, stratify=patient_df["_target"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["_target"]
)

train_ids = set(train_df[PATIENT_COL])
val_ids = set(val_df[PATIENT_COL])
test_ids = set(test_df[PATIENT_COL])

print("Train patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))
print("Train ∩ Validation =", len(train_ids & val_ids))
print("Train ∩ Test =", len(train_ids & test_ids))
print("Validation ∩ Test =", len(val_ids & test_ids))

assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

for name, part in [("Train",train_df),("Validation",val_df),("Test",test_df)]:
    print(name, part["_target"].value_counts().sort_index().to_dict())

## 10 — Clinical Feature Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train = train_df[CLINICAL_FEATURES].copy()
y_train = train_df["_target"].astype(int).copy()
X_val = val_df[CLINICAL_FEATURES].copy()
y_val = val_df["_target"].astype(int).copy()
X_test = test_df[CLINICAL_FEATURES].copy()
y_test = test_df["_target"].astype(int).copy()

numeric_features = [c for c in CLINICAL_FEATURES if pd.api.types.is_numeric_dtype(patient_df[c])]
categorical_features = [c for c in CLINICAL_FEATURES if c not in numeric_features]

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
], remainder="drop", verbose_feature_names_out=False)

# Explicitly fit ONLY on training data.
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()
print("Processed feature count:", len(feature_names))
print("Processed feature names:", feature_names)
print("Shapes:", X_train_proc.shape, X_val_proc.shape, X_test_proc.shape)

## 11 — Class Imbalance Analysis

In [ ]:
train_counts = y_train.value_counts().sort_index()
print("Training distribution:", train_counts.to_dict())
imbalance_ratio = train_counts.max() / train_counts.min()
print("Majority/minority ratio:", round(float(imbalance_ratio), 3))

USE_CLASS_WEIGHT = imbalance_ratio >= 1.5
CLASS_WEIGHT = "balanced" if USE_CLASS_WEIGHT else None
print("Class weighting:", CLASS_WEIGHT)
print("SMOTE: NOT used. Patient records are not synthetically duplicated.")

## 12 — Establish a Baseline

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy.fit(X_train_proc, y_train)
print("DummyClassifier fitted using training data only.")

## 13 — Clinical Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="liblinear",
    class_weight=CLASS_WEIGHT,
    random_state=SEED,
    max_iter=2000
)
logreg.fit(X_train_proc, y_train)

# Optional comparison model. Validation only is used for selection.
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight=CLASS_WEIGHT,
    random_state=SEED,
    min_samples_leaf=2,
    n_jobs=-1
)
rf.fit(X_train_proc, y_train)

print("Models trained: Logistic Regression and Random Forest.")
print("Primary model: Logistic Regression — appropriate for small, interpretable tabular clinical data.")

## 14 — Validation Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

def metrics_dict(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision (RA)": precision_score(y_true, y_pred, zero_division=0),
        "Recall/Sensitivity (RA)": recall_score(y_true, y_pred, zero_division=0),
        "F1 (RA)": f1_score(y_true, y_pred, zero_division=0),
        "Specificity (Non-RA)": specificity,
        "ROC-AUC": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "PR-AUC": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }

def evaluate(model, X, y):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:,1]
    return metrics_dict(y, pred, prob), pred, prob

val_results = {}
for name, model in [("Dummy",dummy),("Logistic Regression",logreg),("Random Forest",rf)]:
    m, _, _ = evaluate(model, X_val_proc, y_val)
    val_results[name] = m

val_table = pd.DataFrame(val_results).T
display(val_table)

# Select by validation balanced accuracy; tie-break using ROC-AUC then simpler Logistic Regression preference.
candidate_models = {"Logistic Regression": logreg, "Random Forest": rf}
selection = val_table.loc[list(candidate_models), ["Balanced Accuracy","ROC-AUC"]].copy()
selection["simplicity_tiebreak"] = [1 if x=="Logistic Regression" else 0 for x in selection.index]
selection = selection.sort_values(["Balanced Accuracy","ROC-AUC","simplicity_tiebreak"], ascending=False)
SELECTED_NAME = selection.index[0]
final_model = candidate_models[SELECTED_NAME]
print("Selected from validation data:", SELECTED_NAME)

In [ ]:
def diagnostic_plots(model, X, y, title):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:,1]
    cm = confusion_matrix(y, pred, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-RA","RA"], yticklabels=["Non-RA","RA"])
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title+" — Confusion Matrix")
    plt.show()

    if len(np.unique(y)) == 2:
        fpr,tpr,_ = roc_curve(y,prob)
        auc = roc_auc_score(y,prob)
        plt.plot(fpr,tpr,label=f"AUC={auc:.3f}")
        plt.plot([0,1],[0,1],"--")
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title(title+" — ROC Curve"); plt.legend(); plt.show()

        p,r,_ = precision_recall_curve(y,prob)
        ap = average_precision_score(y,prob)
        plt.plot(r,p,label=f"PR-AUC={ap:.3f}")
        plt.xlabel("Recall"); plt.ylabel("Precision")
        plt.title(title+" — Precision-Recall Curve"); plt.legend(); plt.show()

diagnostic_plots(final_model, X_val_proc, y_val, "Validation")

## 15 — Final Test Evaluation

In [ ]:
# Test set is touched here for the first and only model-evaluation stage.
test_metrics, test_pred, test_prob = evaluate(final_model, X_test_proc, y_test)
display(pd.DataFrame([test_metrics], index=[SELECTED_NAME]))
diagnostic_plots(final_model, X_test_proc, y_test, "Final Test")
print("IMPORTANT: Do not change model choices based on these test results.")

## 16 — Clinical Model Interpretation

In [ ]:
if SELECTED_NAME == "Logistic Regression":
    importance = pd.DataFrame({
        "feature": feature_names,
        "coefficient": final_model.coef_[0]
    })
    importance["magnitude"] = importance["coefficient"].abs()
    importance = importance.sort_values("magnitude", ascending=False)
    display(importance)

    plot_df = importance.head(20).sort_values("coefficient")
    plt.barh(plot_df["feature"], plot_df["coefficient"])
    plt.axvline(0, linewidth=1)
    plt.title("Logistic Regression Coefficients")
    plt.xlabel("Coefficient")
    plt.show()
    print("Positive coefficients are associated with higher predicted RA probability;")
    print("negative coefficients are associated with lower predicted RA probability.")
    print("These are associations, NOT causal effects.")
else:
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": final_model.feature_importances_
    }).sort_values("importance", ascending=False)
    display(importance)
    plot_df = importance.head(20).sort_values("importance")
    plt.barh(plot_df["feature"], plot_df["importance"])
    plt.title("Random Forest Feature Importance")
    plt.show()
    print("Feature importance is predictive, not causal.")

## 17 — Extract Clinical Features for Future Multimodal Fusion

In [ ]:
# The fusion representation is the processed clinical vector BEFORE the final classifier.
clinical_train_features = np.asarray(X_train_proc, dtype=np.float32)
clinical_val_features = np.asarray(X_val_proc, dtype=np.float32)
clinical_test_features = np.asarray(X_test_proc, dtype=np.float32)

print("Processed clinical feature vectors (preferred future fusion representation):")
print("Train:", clinical_train_features.shape)
print("Validation:", clinical_val_features.shape)
print("Test:", clinical_test_features.shape)

# Prediction probabilities are saved separately; they are NOT the same as the clinical representation.
train_prob = final_model.predict_proba(X_train_proc)[:,1]
val_prob = final_model.predict_proba(X_val_proc)[:,1]
test_prob = final_model.predict_proba(X_test_proc)[:,1]

## 18 — Save Clinical Features and Artifacts

In [ ]:
OUTPUT_DIR = REPO_DIR / "outputs" / "clinical"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR/"clinical_train_features.npy", clinical_train_features)
np.save(OUTPUT_DIR/"clinical_val_features.npy", clinical_val_features)
np.save(OUTPUT_DIR/"clinical_test_features.npy", clinical_test_features)

def make_meta(part, split, probs):
    out = pd.DataFrame({
        "patient_id": part[PATIENT_COL].astype(str).values,
        "true_label": part["_target"].astype(int).values,
        "split": split,
        "ra_probability": probs
    })
    return out

train_meta = make_meta(train_df, "train", train_prob)
val_meta = make_meta(val_df, "validation", val_prob)
test_meta = make_meta(test_df, "test", test_prob)

train_meta.to_csv(OUTPUT_DIR/"clinical_train_metadata.csv", index=False)
val_meta.to_csv(OUTPUT_DIR/"clinical_val_metadata.csv", index=False)
test_meta.to_csv(OUTPUT_DIR/"clinical_test_metadata.csv", index=False)

joblib.dump(final_model, OUTPUT_DIR/"clinical_model.joblib")
joblib.dump(preprocessor, OUTPUT_DIR/"clinical_preprocessor.joblib")

with open(OUTPUT_DIR/"clinical_feature_names.json","w") as f:
    json.dump(feature_names, f, indent=2)

with open(OUTPUT_DIR/"clinical_metrics.json","w") as f:
    json.dump({"selected_model":SELECTED_NAME, "validation":val_results[SELECTED_NAME], "test":test_metrics}, f, indent=2)

config = {
    "seed": SEED,
    "repository": REPO_URL,
    "metadata_file": str(METADATA_PATH.relative_to(REPO_DIR)),
    "patient_column": str(PATIENT_COL),
    "target_column": str(TARGET_COL),
    "clinical_features": list(map(str, CLINICAL_FEATURES)),
    "numeric_features": list(map(str, numeric_features)),
    "categorical_features": list(map(str, categorical_features)),
    "class_weight": CLASS_WEIGHT,
    "selected_model": SELECTED_NAME,
    "split": {"train":0.70,"validation":0.15,"test":0.15}
}
with open(OUTPUT_DIR/"clinical_config.json","w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR/"clinical_test_metrics.csv", index=False)

print("Saved artifacts:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)

## 19 — Verify Feature Alignment With Existing Image Branch

In [ ]:
# Search the repository for image-branch feature metadata/manifests.
# Alignment is checked by patient ID only; row order is NEVER assumed.
alignment_candidates = [
    p for p in all_files
    if p.suffix.lower() in {".csv",".xlsx",".xls",".parquet"}
    and p.resolve() != METADATA_PATH.resolve()
]

def find_patient_column(frame):
    nmap = {c: re.sub(r"[^a-z0-9]+","",str(c).lower()) for c in frame.columns}
    for c,n in nmap.items():
        if n in {"normalizedpatientid","patientid","subjectid","studyid"} or "patientid" in n:
            return c
    return None

found_alignment_table = False
for p in alignment_candidates:
    try:
        t = load_table(p)
        pc = find_patient_column(t)
        if pc is None:
            continue
        found_alignment_table = True
        image_ids = set(t[pc].dropna().astype(str))
        print("\nAlignment candidate:", p.relative_to(REPO_DIR), "| patient key:", pc)
        for split_name, meta in [("train",train_meta),("validation",val_meta),("test",test_meta)]:
            clinical_ids = set(meta["patient_id"].astype(str))
            print(split_name,
                  "both =", len(clinical_ids & image_ids),
                  "| clinical unmatched =", len(clinical_ids - image_ids))
        print("Duplicate patient IDs in candidate:", int(t[pc].astype(str).duplicated().sum()))
    except Exception:
        pass

if not found_alignment_table:
    print("No separate image-feature metadata/manifest with a detectable patient ID was found.")
    print("Clinical files preserve patient IDs, so future alignment can be performed safely once the ResNet18 feature manifest is available.")

## 20 — Model Diagnostics

In [ ]:
# No fake neural-network training curves are generated for Logistic Regression / Random Forest.
plt.hist(test_prob[y_test.values==0], bins=10, alpha=0.6, label="Non-RA")
plt.hist(test_prob[y_test.values==1], bins=10, alpha=0.6, label="RA")
plt.xlabel("Predicted RA probability")
plt.ylabel("Patients")
plt.title("Test Probability Distribution")
plt.legend()
plt.show()

## 21 — Final Summary

In [ ]:
print("="*72)
print("FINAL RA CLINICAL BRANCH SUMMARY")
print("="*72)
print("Dataset")
print("  Total patients:", len(patient_df))
print("  RA patients:", int((patient_df["_target"]==1).sum()))
print("  Non-RA patients:", int((patient_df["_target"]==0).sum()))
print("  Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("\nClinical features")
print("  Original:", CLINICAL_FEATURES)
print("  Processed feature count:", len(feature_names))
print("\nModel")
print("  Selected:", SELECTED_NAME)
print("  Preprocessing: train-fitted imputation + scaling/one-hot encoding")
print("  Imbalance strategy:", CLASS_WEIGHT if CLASS_WEIGHT else "none")
print("\nTest performance")
for k in ["Accuracy","Balanced Accuracy","Precision (RA)","Recall/Sensitivity (RA)",
          "F1 (RA)","Specificity (Non-RA)","ROC-AUC","PR-AUC"]:
    print(f"  {k}: {test_metrics[k]:.4f}")
print("\nFeature extraction")
print("  Train:", clinical_train_features.shape)
print("  Validation:", clinical_val_features.shape)
print("  Test:", clinical_test_features.shape)
print("\nThe clinical branch is now ready for multimodal fusion with the separately trained ResNet18 X-ray branch.")

## Data-Leakage Checklist

- Patient-level splitting occurs before training.
- No patient appears in multiple splits.
- Test data is untouched until final evaluation.
- Imputation, scaling, and one-hot encoding are fitted on training data only.
- Class weighting affects model training only.
- No SMOTE is used.
- Patient IDs and image/file identifiers are excluded from predictive features.
- No X-ray model information is used for clinical model training.
- No multimodal fusion is performed.